<div dir="rtl" style="text-align:right">

# תרגול 9  



התרגול בונה pipeline:

```text
Movie descriptions
→ BoW / TF-IDF
→ PMI / LDA / SVD semantic space
→ PyTorch tensors
→ semantic similarity
```

</div>

<div dir="rtl" style="text-align:right">

## 0. Setup

</div>

In [ ]:
# Core
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Scikit-learn
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.decomposition import TruncatedSVD, LatentDirichletAllocation
from sklearn.metrics.pairwise import cosine_similarity

<div dir="rtl" style="text-align:right">

## 1. קורפוס סרטים

כל שורה היא **Document** אחד — תיאור קצר של סרט.

העמודה `genre` תשמש אותנו רק לצביעה ובדיקה.  
אנחנו לא משתמשים בה כ־target לאימון supervised model.

</div>

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

movies = [
    {
        "title": "Space Rescue",
        "genre": "sci-fi",
        "text": "astronaut space mission planet rescue alien signal spaceship galaxy"
    },
    {
        "title": "Robot City",
        "genre": "sci-fi",
        "text": "robot artificial intelligence city future technology machine human"
    },
    {
        "title": "Galaxy War",
        "genre": "sci-fi",
        "text": "space galaxy war spaceship alien empire planet battle"
    },
    {
        "title": "Love in Paris",
        "genre": "romance",
        "text": "love paris relationship couple romance dinner music heart"
    },
    {
        "title": "Wedding Letters",
        "genre": "romance",
        "text": "wedding love letters family relationship romance heart"
    },
    {
        "title": "Summer Promise",
        "genre": "romance",
        "text": "summer beach love promise couple family memories"
    },
    {
        "title": "Detective Night",
        "genre": "crime",
        "text": "detective murder mystery police investigation suspect crime"
    },
    {
        "title": "Bank Heist",
        "genre": "crime",
        "text": "bank robbery police chase crime thief investigation plan"
    },
    {
        "title": "Hidden Evidence",
        "genre": "crime",
        "text": "evidence detective suspect trial murder police mystery"
    },
    {
        "title": "Dragon Kingdom",
        "genre": "fantasy",
        "text": "dragon kingdom magic sword hero castle battle prophecy"
    },
    {
        "title": "Wizard School",
        "genre": "fantasy",
        "text": "wizard magic school spell student prophecy castle"
    },
    {
        "title": "Forest Quest",
        "genre": "fantasy",
        "text": "hero forest quest magic sword kingdom creature"
    }
]

df = pd.DataFrame(movies)
df

<div dir="rtl" style="text-align:right">

## תרגיל 1 — הבנת הקורפוס

ענו בקצרה:

1. מהו ה־Corpus?
2. מהו Document?
3. מהו Token?
4. מהו Vocabulary?
5. האם `genre` הוא target?
6. האם אנחנו מאמנים supervised model?

</div>

In [ ]:
# TODO:
# df.shape
# df.head()
# df["genre"].value_counts()

<details dir="rtl">
<summary>פתרון מוסתר</summary>

```python
df.shape
df.head()
df["genre"].value_counts()
```

</details>

<div dir="rtl" style="text-align:right">

## 2. EDA קצר

נבדוק כמה מילים יש בכל תיאור וכמה סרטים יש מכל genre.

</div>

In [ ]:
df["n_words"] = df["text"].str.split().str.len()
df[["title", "genre", "n_words"]]

In [ ]:
df["genre"].value_counts().plot(kind="bar")
plt.ylabel("Number of movies")
plt.title("Movies per Genre")
plt.show()

<div dir="rtl" style="text-align:right">

## תרגיל 2 — שאלות EDA

ענו:

1. האם הקורפוס מאוזן מבחינת genres?
2. האם כל המסמכים באורך דומה?
3. למה קורפוס קטן כזה טוב לתרגול?
4. למה קורפוס קטן כזה בעייתי למחקר אמיתי?

</div>

In [ ]:
# TODO:
# כתבו תשובות קצרות במרקדאון או בהערות קוד.

<details dir="rtl">
<summary>פתרון מוסתר</summary>

```python
df["genre"].value_counts()
df[["title", "genre", "n_words"]].sort_values("n_words", ascending=False)
```

</details>

<div dir="rtl" style="text-align:right">

## 3. Bag of Words

נתחיל מייצוג פשוט: ספירת מילים.

</div>

In [ ]:
vec = CountVectorizer()
X_bow = vec.fit_transform(df["text"])
terms = vec.get_feature_names_out()

dtm = pd.DataFrame(
    X_bow.toarray(),
    columns=terms,
    index=df["title"]
)

dtm.head()

<div dir="rtl" style="text-align:right">

## תרגיל 3 — מילים נפוצות

חשבו את 15 המילים הכי נפוצות בקורפוס והציגו אותן ב־Bar Plot.

</div>

In [ ]:
# TODO:
# term_counts = ...
# top_terms = ...
# display(top_terms.head(15))
# plot

<details dir="rtl">
<summary>פתרון מוסתר</summary>

```python
term_counts = np.asarray(X_bow.sum(axis=0)).ravel()

top_terms = pd.DataFrame({
    "term": terms,
    "count": term_counts
}).sort_values("count", ascending=False)

display(top_terms.head(15))

top_terms.head(15).plot(x="term", y="count", kind="bar", legend=False)
plt.ylabel("Count")
plt.title("Top Terms")
plt.show()
```

</details>

<div dir="rtl" style="text-align:right">

## 4. PMI / PPMI — מילים שמופיעות יחד

PMI עוזר לזהות צמדי מילים שמופיעים יחד יותר ממה שהיינו מצפים במקרה.

נחשב co-occurrence ברמת מסמך: שתי מילים מופיעות יחד אם הן מופיעות באותו מסמך.

</div>

In [ ]:
X_bin = (X_bow.toarray() > 0).astype(int)

cooc = X_bin.T @ X_bin
cooc_df = pd.DataFrame(cooc, index=terms, columns=terms)

cooc_df.head()

<div dir="rtl" style="text-align:right">

## תרגיל 4 — חישוב PPMI

השלימו את חישוב ה־PMI וה־PPMI, ואז הציגו את 15 צמדי המילים עם הציון הגבוה ביותר.

</div>

In [ ]:
# TODO:
# total_docs = ...
# word_prob = X_bin.sum(axis=0) / total_docs
# cooc_prob = cooc / total_docs
# pmi = ...
# ppmi = ...

# TODO:
# צרו DataFrame של pairs ומיינו לפי ppmi
pairs = []

for i in range(len(terms)):
    for j in range(i + 1, len(terms)):
        pairs.append({
            "term_1": terms[i],
            "term_2": terms[j],
            "ppmi": ppmi[i, j]
        })

ppmi_pairs = pd.DataFrame(pairs).sort_values("ppmi", ascending=False)

ppmi_pairs.head(15)

<details dir="rtl">
<summary>פתרון מוסתר</summary>

```python
total_docs = X_bin.shape[0]

word_prob = X_bin.sum(axis=0) / total_docs
cooc_prob = cooc / total_docs

pmi = np.zeros_like(cooc_prob, dtype=float)

for i in range(len(terms)):
    for j in range(len(terms)):
        if cooc_prob[i, j] > 0:
            pmi[i, j] = np.log2(cooc_prob[i, j] / (word_prob[i] * word_prob[j]))
        else:
            pmi[i, j] = 0

ppmi = np.maximum(pmi, 0)

pairs = []

for i in range(len(terms)):
    for j in range(i + 1, len(terms)):
        pairs.append({
            "term_1": terms[i],
            "term_2": terms[j],
            "ppmi": ppmi[i, j]
        })

ppmi_pairs = pd.DataFrame(pairs).sort_values("ppmi", ascending=False)

ppmi_pairs.head(15)
```

</details>

<div dir="rtl" style="text-align:right">

## שאלות חשיבה על PMI

ענו:

1. אילו צמדי מילים קיבלו PPMI גבוה?
2. האם הם מתאימים לז׳אנרים?
3. מה ההבדל בין מילה נפוצה לבין צמד מילים משמעותי?
4. איך זה מחבר אותנו לרעיון של GloVe?

</div>

In [ ]:
# TODO:
# כתבו תשובות קצרות.

<details dir="rtl">
<summary>פתרון מוסתר</summary>

```python
ppmi_pairs.head(20)
```

</details>

<div dir="rtl" style="text-align:right">

## 5. TF-IDF ו־SVD / LSA

נעבור מ־BoW ל־TF-IDF, ואז נשתמש ב־TruncatedSVD כדי לבנות מרחב סמנטי מוקטן.

</div>

In [ ]:
tfidf_vec = TfidfVectorizer()
X_tfidf = tfidf_vec.fit_transform(df["text"])

tfidf_terms = tfidf_vec.get_feature_names_out()

print("TF-IDF shape:", X_tfidf.shape)
print("Vocabulary size:", len(tfidf_terms))

In [ ]:
svd = TruncatedSVD(n_components=2, random_state=42)
X_lsa_2d = svd.fit_transform(X_tfidf)

print("LSA shape:", X_lsa_2d.shape)
print("Explained variance ratio:", svd.explained_variance_ratio_)

<div dir="rtl" style="text-align:right">

## תרגיל 5 — ויזואליזציה של מרחב SVD

ציירו את הסרטים במרחב דו־ממדי:

- ציר X: רכיב SVD ראשון.
- ציר Y: רכיב SVD שני.
- צבע לפי `genre`.
- הוסיפו שמות סרטים על הגרף.

</div>

In [ ]:
# TODO:
# plt.figure(figsize=(10, 7))

# for genre in df["genre"].unique():
#     mask = df["genre"] == genre
#     plt.scatter(
#         X_lsa_2d[mask, 0],
#         X_lsa_2d[mask, 1],
#         label=genre
#     )

# for i, title in enumerate(df["title"]):
#     plt.text(X_lsa_2d[i, 0], X_lsa_2d[i, 1], title, fontsize=8)

# plt.xlabel("SVD component 1")
# plt.ylabel("SVD component 2")
# plt.title("Movies in LSA Semantic Space")
# plt.legend()
# plt.show()

<details dir="rtl">
<summary>פתרון מוסתר</summary>

```python
plt.figure(figsize=(10, 7))

for genre in df["genre"].unique():
    mask = df["genre"] == genre
    plt.scatter(
        X_lsa_2d[mask, 0],
        X_lsa_2d[mask, 1],
        label=genre
    )

for i, title in enumerate(df["title"]):
    plt.text(X_lsa_2d[i, 0], X_lsa_2d[i, 1], title, fontsize=8)

plt.xlabel("SVD component 1")
plt.ylabel("SVD component 2")
plt.title("Movies in LSA Semantic Space")
plt.legend()
plt.show()
```

</details>

<div dir="rtl" style="text-align:right">

## תרגיל 6 — פרשנות רכיבי SVD

בדקו אילו מילים מקבלות משקל גבוה בכל רכיב SVD.

</div>

In [ ]:
# TODO:
# components = svd.components_


<details dir="rtl">
<summary>פתרון מוסתר</summary>

```python
components = svd.components_

for comp_idx, comp in enumerate(components):
    top_idx = np.argsort(comp)[-8:][::-1]
    print(f"\nComponent {comp_idx + 1}:")
    for idx in top_idx:
        print(tfidf_terms[idx], round(comp[idx], 3))
```

</details>

<div dir="rtl" style="text-align:right">

## 6. Semantic Search במרחב SVD

נחפש סרטים לפי שאילתה, אבל נשווה במרחב SVD המוקטן.

</div>

In [ ]:
query = "alien spaceship and planet mission"

q_tfidf = tfidf_vec.transform([query])
q_lsa = svd.transform(q_tfidf)

scores = cosine_similarity(q_lsa, X_lsa_2d).ravel()

df.assign(score=scores).sort_values("score", ascending=False)[
    ["title", "genre", "text", "score"]
].head(5)

<div dir="rtl" style="text-align:right">

## תרגיל 7 — בדיקת שאילתות

בדקו את השאילתות הבאות:

```python
queries = [
    "alien spaceship planet",
    "love wedding couple",
    "detective police murder",
    "magic kingdom sword",
    "robot artificial intelligence future"
]
```

עבור כל שאילתה הציגו Top 3 תוצאות.

</div>

In [ ]:
# TODO:
queries = [
    "alien spaceship planet",
    "love wedding couple",
    "detective police murder",
    "magic kingdom sword",
    "robot artificial intelligence future"
]
# for query in queries:
#     ...

<details dir="rtl">
<summary>פתרון מוסתר</summary>

```python
queries = [
    "alien spaceship planet",
    "love wedding couple",
    "detective police murder",
    "magic kingdom sword",
    "robot artificial intelligence future"
]

for query in queries:
    q_tfidf = tfidf_vec.transform([query])
    q_lsa = svd.transform(q_tfidf)
    scores = cosine_similarity(q_lsa, X_lsa_2d).ravel()

    print("\nQuery:", query)
    display(
        df.assign(score=scores)
          .sort_values("score", ascending=False)
          [["title", "genre", "score"]]
          .head(3)
    )
```

</details>

<div dir="rtl" style="text-align:right">

## 7. LDA Topic Modeling

LDA מניח שכל מסמך הוא תערובת של topics, וכל topic הוא התפלגות על מילים.

</div>

In [ ]:
count_vec = CountVectorizer()
X_counts = count_vec.fit_transform(df["text"])
count_terms = count_vec.get_feature_names_out()

lda = LatentDirichletAllocation(
    n_components=4,
    random_state=42,
    learning_method="batch"
)

doc_topic = lda.fit_transform(X_counts)

print("Document-topic matrix shape:", doc_topic.shape)

<div dir="rtl" style="text-align:right">

## תרגיל 8 — הצגת מילים מובילות בכל Topic

הציגו לכל topic את 8 המילים עם המשקל הגבוה ביותר.

</div>

In [ ]:
# TODO:
# for topic_idx, topic in enumerate(lda.components_):
#     ...

<details dir="rtl">
<summary>פתרון מוסתר</summary>

```python
for topic_idx, topic in enumerate(lda.components_):
    top_idx = topic.argsort()[-8:][::-1]
    top_words = [count_terms[i] for i in top_idx]
    print(f"Topic {topic_idx}: {top_words}")
```

</details>

<div dir="rtl" style="text-align:right">

## תרגיל 9 — Topic דומיננטי לכל סרט

הוסיפו ל־DataFrame עמודה בשם `dominant_topic`.

לאחר מכן השוו בין `genre` לבין `dominant_topic`.

</div>

In [ ]:
# TODO:
# df["dominant_topic"] = ...
# df[["title", "genre", "dominant_topic"]]
# pd.crosstab(...)

<details dir="rtl">
<summary>פתרון מוסתר</summary>

```python
df["dominant_topic"] = doc_topic.argmax(axis=1)

display(df[["title", "genre", "dominant_topic"]])

pd.crosstab(df["dominant_topic"], df["genre"])
```

</details>

<div dir="rtl" style="text-align:right">

## שאלות השוואה — LDA מול SVD

ענו:

1. האם topics של LDA דומים לז׳אנרים?
2. האם הם דומים לרכיבי SVD?
3. מה ההבדל בין component של SVD לבין topic של LDA?
4. האם כל מסמך חייב להשתייך ל־topic אחד בלבד?
5. למה LDA מתאים לפרשנות?

</div>

In [ ]:
# TODO:
# כתבו תשובות קצרות.

<details dir="rtl">
<summary>פתרון מוסתר</summary>

```python
display(pd.crosstab(df["dominant_topic"], df["genre"]))

components = svd.components_
for comp_idx, comp in enumerate(components):
    top_idx = np.argsort(comp)[-8:][::-1]
    print(f"\nSVD Component {comp_idx + 1}:")
    print([tfidf_terms[idx] for idx in top_idx])

for topic_idx, topic in enumerate(lda.components_):
    top_idx = topic.argsort()[-8:][::-1]
    print(f"\nLDA Topic {topic_idx}:")
    print([count_terms[i] for i in top_idx])
```

</details>

<div dir="rtl" style="text-align:right">

## 8. מעבר ל־PyTorch Tensors

ניקח את הייצוגים הסמנטיים שקיבלנו מ־SVD ונמיר אותם ל־PyTorch Tensor.

</div>

In [ ]:
try:
    import torch
except ImportError as e:
    raise ImportError(
        "PyTorch is not installed in this environment. "
        "In Colab, run: !pip install torch"
    ) from e

if torch.cuda.is_available():
    device = "cuda"
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"

device

In [ ]:
X_tensor = torch.tensor(X_lsa_2d, dtype=torch.float32, device=device)

print("Tensor:")
print(X_tensor)
print("Shape:", X_tensor.shape)
print("Dtype:", X_tensor.dtype)
print("Device:", X_tensor.device)
print("Number of dimensions:", X_tensor.ndim)

<div dir="rtl" style="text-align:right">

## תרגיל 10 — Tensor Basics

ענו:

1. מה ה־shape של `X_tensor`?
2. מה מייצגת כל שורה?
3. מה מייצגת כל עמודה?
4. למה `dtype=torch.float32` מתאים כאן?
5. מה ההבדל בין CPU, CUDA ו־MPS?

</div>

In [ ]:
# TODO:
# כתבו תשובות קצרות.

<details dir="rtl">
<summary>פתרון מוסתר</summary>

```python
print("Shape:", X_tensor.shape)
print("Dtype:", X_tensor.dtype)
print("Device:", X_tensor.device)
print("Number of dimensions:", X_tensor.ndim)
print("First row:", X_tensor[0])
```

</details>

<div dir="rtl" style="text-align:right">

## 9. Cosine Similarity ב־PyTorch

נחשב מטריצת דמיון בין כל הסרטים בעזרת PyTorch.

</div>

In [ ]:
X_norm = X_tensor / X_tensor.norm(dim=1, keepdim=True)

similarity_matrix = X_norm @ X_norm.T

sim_df = pd.DataFrame(
    similarity_matrix.detach().cpu().numpy(),
    index=df["title"],
    columns=df["title"]
)

sim_df.round(2)

<div dir="rtl" style="text-align:right">

## תרגיל 11 — Heatmap של דמיון סרטים

ציירו heatmap של מטריצת הדמיון.

</div>

In [ ]:
# TODO:
# plt.figure(figsize=(9, 8))
# plt.imshow(similarity_matrix.detach().cpu().numpy())
# plt.xticks(range(len(df)), df["title"], rotation=90)
# plt.yticks(range(len(df)), df["title"])
# plt.colorbar()
# plt.title("Movie Similarity Matrix")
# plt.tight_layout()
# plt.show()

<details dir="rtl">
<summary>פתרון מוסתר</summary>

```python
plt.figure(figsize=(9, 8))
plt.imshow(similarity_matrix.detach().cpu().numpy())
plt.xticks(range(len(df)), df["title"], rotation=90)
plt.yticks(range(len(df)), df["title"])
plt.colorbar()
plt.title("Movie Similarity Matrix")
plt.tight_layout()
plt.show()
```

</details>

<div dir="rtl" style="text-align:right">

## תרגיל 12 — פונקציה למציאת סרטים דומים

כתבו פונקציה:

```python
most_similar_movie(title, top_n=5)
```

הפונקציה תקבל שם סרט ותחזיר את הסרטים הכי דומים לו לפי מטריצת הדמיון.

</div>

In [ ]:
# TODO:
# def most_similar_movie(title, top_n=5):
#     ...
#
# most_similar_movie("Space Rescue")

<details dir="rtl">
<summary>פתרון מוסתר</summary>

```python
def most_similar_movie(title, top_n=5):
    idx = df.index[df["title"] == title][0]

    scores = similarity_matrix[idx].detach().cpu().numpy()

    result = df[["title", "genre"]].copy()
    result["score"] = scores

    return result.sort_values("score", ascending=False).head(top_n)

most_similar_movie("Space Rescue")
```

</details>

<div dir="rtl" style="text-align:right">

## שאלות סיכום

ענו בקצרה:

1. מה ההבדל בין BoW, TF-IDF, SVD, LDA ו־GloVe?
2. מה SVD מוצא במטריצת TF-IDF?
3. למה קוראים לזה LSA / LSI?
4. מה LDA נותן ש־SVD לא נותן?
5. למה PyTorch Tensor מתאים לייצוג כזה?
6. האם השתמשנו כאן ב־Deep Learning אמיתי?
7. מה חסר כדי להפוך את זה לאימון רשת נוירונים?
8. מה הקשר בין co-occurrence לבין embeddings?

</div>

<details dir="rtl">
<summary>פתרון מוסתר</summary>

```python
# Suggested checks for summary discussion:
print("BoW shape:", X_bow.shape)
print("TF-IDF shape:", X_tfidf.shape)
print("SVD representation shape:", X_lsa_2d.shape)
print("LDA document-topic shape:", doc_topic.shape)
print("PyTorch tensor shape:", X_tensor.shape)
print("Used supervised labels for fitting?", False)
```

</details>